In [32]:
%pip install torchmetrics pycocotools

Note: you may need to restart the kernel to use updated packages.


In [33]:
import os
import torch
import torchvision
from torchvision.models.detection import FasterRCNN
from torchvision.models.detection.rpn import AnchorGenerator
import json
from PIL import Image
from torch.utils.data import Dataset, Subset, DataLoader
from sklearn.model_selection import train_test_split
from torchvision import transforms as T
from torchvision.models.detection import fasterrcnn_resnet50_fpn, FasterRCNN_ResNet50_FPN_Weights
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
import time
import torch.backends.cudnn as cudnn
cudnn.benchmark = True
import albumentations as A
import cv2
from albumentations.pytorch import ToTensorV2
from torch.utils.data import WeightedRandomSampler
import numpy as np



DataSet 클래스

In [34]:
class PillDataset(Dataset):
    def __init__(self, img_dir, json_dir, transforms=None):
        self.img_dir = img_dir
        self.json_dir = json_dir
        self.transforms = transforms
        self.img_files = [f for f in os.listdir(img_dir) if f.endswith(('.png', '.jpg'))]

    def __getitem__(self, idx):
        img_name = self.img_files[idx]
        img_path = os.path.join(self.img_dir, img_name)
        json_path = os.path.join(
            self.json_dir,
            os.path.splitext(img_name)[0] + ".json"
        )

        # 이미지 로드
        img = Image.open(img_path).convert("RGB")
        img = np.array(img)

        boxes = []
        labels = []

        # JSON 로드
        if os.path.exists(json_path):
            with open(json_path, 'r', encoding='utf-8') as f:
                data = json.load(f)

            for ann in data.get("annotations", []):
                bbox = ann["bbox"]  # [x, y, w, h]
                category_id = ann["category_id"]

                x, y, w, h = bbox
                boxes.append([x, y, x + w, y + h])  # xyxy로 변환
                labels.append(category_id)

        # tensor 변환 전 준비
        boxes = torch.as_tensor(boxes, dtype=torch.float32)
        labels = torch.as_tensor(labels, dtype=torch.int64)

        target = {
            "boxes": boxes,
            "labels": labels,
            "image_id": torch.tensor([idx])
        }

        # Albumentations 적용
        if self.transforms:
            transformed = self.transforms(
                image=img,
                bboxes=boxes.numpy(),
                class_labels=labels.tolist()
            )

            img = transformed["image"].float() / 255.0
            boxes = torch.tensor(transformed["bboxes"], dtype=torch.float32)
            labels = torch.tensor(transformed["class_labels"], dtype=torch.int64)

            target["boxes"] = boxes
            target["labels"] = labels

        # 빈 bbox 처리 (🔥 중요)
        if len(target["boxes"]) == 0:
            target["boxes"] = torch.zeros((0, 4), dtype=torch.float32)
            target["labels"] = torch.zeros((0,), dtype=torch.int64)

        return img, target

    def __len__(self):
        return len(self.img_files)

DataLoader 설정

In [35]:
import pickle

def get_transform():
    return A.Compose([
        A.LongestMaxSize(max_size=800),
        ToTensorV2(),
    ], bbox_params=A.BboxParams(
        format='pascal_voc', 
        label_fields=['class_labels'],
        clip=False,
        min_area = 2.0,
        min_visibility=0.1
    ))

full_dataset = PillDataset(
    img_dir=r'C:\Users\thsdu\Desktop\beg_project\Datas\fixed_data\aug_sampling_data\images',
    json_dir=r'C:\Users\thsdu\Desktop\beg_project\Datas\fixed_data\aug_sampling_data\annotations',
    transforms=get_transform()
)

indices = list(range(len(full_dataset)))
train_indices, val_indices = train_test_split(indices, test_size=0.2, random_state=42)

train_dataset = Subset(full_dataset, train_indices)
val_dataset = Subset(full_dataset, val_indices)

class_counts = {
    0:149, 1:231, 2:274, 3:135, 4:226, 5:342, 6:224, 7:308,
    8:279, 9:335, 10:330, 11:256, 12:220, 13:915, 14:557,
    15:814, 16:794, 17:248, 18:304, 19:313, 20:317, 21:332,
    22:201, 23:321, 24:293, 25:574, 26:344, 27:283, 28:201,
    29:250, 30:340, 31:245, 32:202, 33:329, 34:277, 35:338,
    36:309, 37:298, 38:194, 39:158, 40:329, 41:187, 42:188,
    43:564, 44:319, 45:280, 46:303, 47:304, 48:191, 49:198,
    50:301, 51:319, 52:570, 53:557, 54:258, 55:315
}

# inverse frequency
class_weights = {k: 1.0 / v for k, v in class_counts.items()}

def get_image_weight(target):
    labels = target['labels'].tolist()
    
    if len(labels) == 0:
        return 0.1
    
    weights = [class_weights[l] for l in labels]
    return max(weights)

if os.path.exists("sample_weights.pkl"):
    with open("sample_weights.pkl", "rb") as f:
        sample_weights = pickle.load(f)
else:
    sample_weights = []
    for i in range(len(train_dataset)):
        _, target = train_dataset[i]
        w = get_image_weight(target)
        sample_weights.append(w)

    with open("sample_weights.pkl", "wb") as f:
        pickle.dump(sample_weights, f)
        
sampler = WeightedRandomSampler(
    sample_weights,
    num_samples=len(sample_weights),
    replacement=True
)




def collate_fn(batch):
    return tuple(zip(*batch))


train_loader = DataLoader(
    train_dataset, 
    batch_size=4,          
    shuffle=False, #sampler 쓰기때문에
    sampler=sampler,
    num_workers=0,         
    collate_fn=collate_fn  
)

val_loader = DataLoader(
    val_dataset, 
    batch_size=2, 
    shuffle=False, 
    num_workers=0, 
    collate_fn=collate_fn
)

In [36]:
from torchmetrics.detection.mean_ap import MeanAveragePrecision
from tqdm import tqdm

def evaluate(model, data_loader, device):
    model.eval() # 평가 모드 (Loss 계산이 아니라 예측값을 내뱉음)
    metric = MeanAveragePrecision(iou_type="bbox")
    
    with torch.no_grad():
        for images, targets in tqdm(data_loader, desc="Evaluating"):
            images = list(img.to(device) for img in images)
            outputs = model(images) # 예측 결과 [{boxes, scores, labels}, ...]
            
            # targets를 device로 옮기고 Metric 형식에 맞게 변환
            res_targets = [{k: v.to(device) for k, v in t.items()} for t in targets]
            filtered_outputs = []

            for o in outputs:
                boxes = o['boxes']
                scores = o['scores']
                labels = o['labels']

                valid_boxes = []
                valid_scores = []
                valid_labels = []

                for box, score, label in zip(boxes, scores, labels):
                    x1, y1, x2, y2 = box

                    if x2 <= x1 or y2 <= y1:
                        continue
                    
                    if score < 0.05:
                        continue
                    valid_boxes.append(box)
                    valid_scores.append(score)
                    valid_labels.append(label)

                if len(valid_boxes) == 0:
                    filtered_outputs.append({
                        "boxes": torch.zeros((0, 4), device=device),
                        "scores": torch.zeros((0,), device=device),
                        "labels": torch.zeros((0,), dtype=torch.int64, device=device)
                    })
                else:
                    filtered_outputs.append({
                        "boxes": torch.stack(valid_boxes),
                        "scores": torch.stack(valid_scores),
                        "labels": torch.stack(valid_labels),
                    })

            res_outputs = filtered_outputs
            
            metric.update(res_outputs, res_targets)

    # 모든 배치 계산 후 최종 mAP 산출
    result = metric.compute()
    
    print("Validation Performance")
    print(f"mAP (IoU=0.50:0.95): {result['map']:.4f}")
    print(f"mAP@50: {result['map_50']:.4f}")
    print(f"mAP@75: {result['map_75']:.4f}")

    
    return result

In [37]:


device = 'cuda' if torch.cuda.is_available() else 'cpu'

weights = FasterRCNN_ResNet50_FPN_Weights.DEFAULT
model = fasterrcnn_resnet50_fpn(weights=weights)

# 배경 포함 클래스수
num_classes = 57 
in_features = model.roi_heads.box_predictor.cls_score.in_features
model.roi_heads.box_predictor = torchvision.models.detection.faster_rcnn.FastRCNNPredictor(in_features, num_classes=num_classes)

model.to(device)

params = [p for p in model.parameters() if p.requires_grad]
optimizer = torch.optim.SGD(params, lr=0.0005, momentum=0.9, weight_decay=0.0005)

def train_one_epoch(model, optimizer, data_loader, device, epoch):
    model.train()
    total_loss = 0
    start_time = time.time()

    for i, batch in enumerate(data_loader):
        images, targets = batch
        images = list(image.to(device) for image in images)
        targets = [{k: v.to(device) for k, v in t.items()} for t in targets]

        loss_dict = model(images, targets)
        

        losses = sum(loss for loss in loss_dict.values())

        # 3. 역전파 및 최적화 (Backward & Optimizer Step)
        optimizer.zero_grad()
        losses.backward()
        # 그래디언트가 과도하게 커지지 않도록 제한검
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        total_loss += losses.item()

        # 10번의 반복마다 현재 상황 출력
        if i % 50 == 0:
            print(f"Epoch [{epoch}] Iter [{i}/{len(data_loader)}] "
                  f"Loss: {losses.item():.4f} "
                  f"(RPN_Obj: {loss_dict['loss_objectness']:.3f}, "
                  f"RPN_Reg: {loss_dict['loss_rpn_box_reg']:.3f}, "
                  f"Box: {loss_dict['loss_box_reg']:.3f})")

    avg_loss = total_loss / len(data_loader)
    elapsed = time.time() - start_time
    print(f"==> Epoch {epoch} 평균 Loss: {avg_loss:.4f} | 소요 시간: {elapsed:.1f}s")
    return avg_loss


num_epochs = 15
loss_history = []
print(f"Model is on: {next(model.parameters()).device}")

scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=5, gamma=0.3)

for epoch in range(num_epochs):
    epoch_loss = train_one_epoch(model, optimizer, train_loader, device, epoch)
    loss_history.append(epoch_loss)

    scheduler.step()

    val_stats = evaluate(model, val_loader, device)

    # 5 에폭마다 가중치 저장 (백업용)
    if epoch % 5 == 0:
         torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'mAP': val_stats['map'],
        }, f"pill_model_ep{epoch}.pth")

print("모든 학습이 완료됨")

Model is on: cuda:0
Epoch [0] Iter [0/1566] Loss: 4.5903 (RPN_Obj: 0.003, RPN_Reg: 0.005, Box: 0.489)
Epoch [0] Iter [50/1566] Loss: 1.0132 (RPN_Obj: 0.003, RPN_Reg: 0.003, Box: 0.338)
Epoch [0] Iter [100/1566] Loss: 1.1244 (RPN_Obj: 0.000, RPN_Reg: 0.002, Box: 0.492)
Epoch [0] Iter [150/1566] Loss: 0.9403 (RPN_Obj: 0.002, RPN_Reg: 0.001, Box: 0.416)
Epoch [0] Iter [200/1566] Loss: 0.8232 (RPN_Obj: 0.000, RPN_Reg: 0.001, Box: 0.362)
Epoch [0] Iter [250/1566] Loss: 0.8778 (RPN_Obj: 0.000, RPN_Reg: 0.002, Box: 0.400)
Epoch [0] Iter [300/1566] Loss: 0.9934 (RPN_Obj: 0.000, RPN_Reg: 0.001, Box: 0.440)
Epoch [0] Iter [350/1566] Loss: 0.6385 (RPN_Obj: 0.000, RPN_Reg: 0.001, Box: 0.274)
Epoch [0] Iter [400/1566] Loss: 0.7594 (RPN_Obj: 0.000, RPN_Reg: 0.001, Box: 0.332)
Epoch [0] Iter [450/1566] Loss: 0.6812 (RPN_Obj: 0.000, RPN_Reg: 0.001, Box: 0.303)
Epoch [0] Iter [500/1566] Loss: 0.8040 (RPN_Obj: 0.000, RPN_Reg: 0.001, Box: 0.356)
Epoch [0] Iter [550/1566] Loss: 0.6310 (RPN_Obj: 0.000, RPN

Evaluating: 100%|██████████| 783/783 [01:49<00:00,  7.12it/s]


Validation Performance
mAP (IoU=0.50:0.95): 0.4798
mAP@50: 0.8206
mAP@75: 0.5109
Epoch [1] Iter [0/1566] Loss: 0.4071 (RPN_Obj: 0.011, RPN_Reg: 0.002, Box: 0.198)
Epoch [1] Iter [50/1566] Loss: 0.3734 (RPN_Obj: 0.000, RPN_Reg: 0.002, Box: 0.168)
Epoch [1] Iter [100/1566] Loss: 0.3319 (RPN_Obj: 0.000, RPN_Reg: 0.002, Box: 0.174)
Epoch [1] Iter [150/1566] Loss: 0.3866 (RPN_Obj: 0.000, RPN_Reg: 0.002, Box: 0.212)
Epoch [1] Iter [200/1566] Loss: 0.2906 (RPN_Obj: 0.009, RPN_Reg: 0.001, Box: 0.153)
Epoch [1] Iter [250/1566] Loss: 0.3321 (RPN_Obj: 0.005, RPN_Reg: 0.004, Box: 0.201)
Epoch [1] Iter [300/1566] Loss: 0.3824 (RPN_Obj: 0.000, RPN_Reg: 0.001, Box: 0.223)
Epoch [1] Iter [350/1566] Loss: 0.4019 (RPN_Obj: 0.001, RPN_Reg: 0.001, Box: 0.237)
Epoch [1] Iter [400/1566] Loss: 0.3246 (RPN_Obj: 0.017, RPN_Reg: 0.004, Box: 0.208)
Epoch [1] Iter [450/1566] Loss: 0.2502 (RPN_Obj: 0.000, RPN_Reg: 0.001, Box: 0.138)
Epoch [1] Iter [500/1566] Loss: 0.3312 (RPN_Obj: 0.000, RPN_Reg: 0.002, Box: 0.188

Evaluating: 100%|██████████| 783/783 [01:34<00:00,  8.24it/s]


Validation Performance
mAP (IoU=0.50:0.95): 0.6565
mAP@50: 0.9307
mAP@75: 0.8399
Epoch [2] Iter [0/1566] Loss: 0.4169 (RPN_Obj: 0.000, RPN_Reg: 0.002, Box: 0.233)
Epoch [2] Iter [50/1566] Loss: 0.2101 (RPN_Obj: 0.000, RPN_Reg: 0.001, Box: 0.137)
Epoch [2] Iter [100/1566] Loss: 0.3071 (RPN_Obj: 0.000, RPN_Reg: 0.002, Box: 0.140)
Epoch [2] Iter [150/1566] Loss: 0.2159 (RPN_Obj: 0.000, RPN_Reg: 0.001, Box: 0.147)
Epoch [2] Iter [200/1566] Loss: 0.1845 (RPN_Obj: 0.000, RPN_Reg: 0.001, Box: 0.125)
Epoch [2] Iter [250/1566] Loss: 0.2840 (RPN_Obj: 0.002, RPN_Reg: 0.003, Box: 0.184)
Epoch [2] Iter [300/1566] Loss: 0.3371 (RPN_Obj: 0.000, RPN_Reg: 0.001, Box: 0.170)
Epoch [2] Iter [350/1566] Loss: 0.1753 (RPN_Obj: 0.000, RPN_Reg: 0.000, Box: 0.084)
Epoch [2] Iter [400/1566] Loss: 0.2145 (RPN_Obj: 0.000, RPN_Reg: 0.002, Box: 0.155)
Epoch [2] Iter [450/1566] Loss: 0.2280 (RPN_Obj: 0.000, RPN_Reg: 0.003, Box: 0.140)
Epoch [2] Iter [500/1566] Loss: 0.2935 (RPN_Obj: 0.000, RPN_Reg: 0.001, Box: 0.173

...

Validation Performance
mAP (IoU=0.50:0.95): 0.8612
mAP@50: 0.9688
mAP@75: 0.9636
Epoch [13] Iter [0/1566] Loss: 0.1145 (RPN_Obj: 0.000, RPN_Reg: 0.001, Box: 0.068)
Epoch [13] Iter [50/1566] Loss: 0.1100 (RPN_Obj: 0.000, RPN_Reg: 0.001, Box: 0.069)
Epoch [13] Iter [100/1566] Loss: 0.0792 (RPN_Obj: 0.000, RPN_Reg: 0.000, Box: 0.050)
Epoch [13] Iter [150/1566] Loss: 0.1030 (RPN_Obj: 0.000, RPN_Reg: 0.001, Box: 0.063)
Epoch [13] Iter [200/1566] Loss: 0.0900 (RPN_Obj: 0.000, RPN_Reg: 0.001, Box: 0.057)
Epoch [13] Iter [250/1566] Loss: 0.1662 (RPN_Obj: 0.002, RPN_Reg: 0.001, Box: 0.076)
Epoch [13] Iter [300/1566] Loss: 0.0630 (RPN_Obj: 0.000, RPN_Reg: 0.000, Box: 0.037)
Epoch [13] Iter [350/1566] Loss: 0.1152 (RPN_Obj: 0.000, RPN_Reg: 0.001, Box: 0.076)
Epoch [13] Iter [400/1566] Loss: 0.1412 (RPN_Obj: 0.000, RPN_Reg: 0.001, Box: 0.082)
Epoch [13] Iter [450/1566] Loss: 0.1158 (RPN_Obj: 0.000, RPN_Reg: 0.001, Box: 0.054)
Epoch [13] Iter [500/1566] Loss: 0.0937 (RPN_Obj: 0.000, RPN_Reg: 0.001,

Evaluating: 100%|██████████| 783/783 [01:18<00:00,  9.91it/s]


Validation Performance
mAP (IoU=0.50:0.95): 0.8635
mAP@50: 0.9690
mAP@75: 0.9642
Epoch [14] Iter [0/1566] Loss: 0.1002 (RPN_Obj: 0.000, RPN_Reg: 0.000, Box: 0.053)
Epoch [14] Iter [50/1566] Loss: 0.1051 (RPN_Obj: 0.004, RPN_Reg: 0.001, Box: 0.062)
Epoch [14] Iter [100/1566] Loss: 0.0775 (RPN_Obj: 0.000, RPN_Reg: 0.001, Box: 0.050)
Epoch [14] Iter [150/1566] Loss: 0.0896 (RPN_Obj: 0.000, RPN_Reg: 0.000, Box: 0.053)
Epoch [14] Iter [200/1566] Loss: 0.1251 (RPN_Obj: 0.000, RPN_Reg: 0.001, Box: 0.083)
Epoch [14] Iter [250/1566] Loss: 0.1028 (RPN_Obj: 0.000, RPN_Reg: 0.001, Box: 0.058)
Epoch [14] Iter [300/1566] Loss: 0.0666 (RPN_Obj: 0.001, RPN_Reg: 0.003, Box: 0.043)
Epoch [14] Iter [350/1566] Loss: 0.0683 (RPN_Obj: 0.000, RPN_Reg: 0.001, Box: 0.037)
Epoch [14] Iter [400/1566] Loss: 0.1254 (RPN_Obj: 0.001, RPN_Reg: 0.001, Box: 0.057)
Epoch [14] Iter [450/1566] Loss: 0.0967 (RPN_Obj: 0.000, RPN_Reg: 0.001, Box: 0.064)
Epoch [14] Iter [500/1566] Loss: 0.1034 (RPN_Obj: 0.000, RPN_Reg: 0.002,

Evaluating: 100%|██████████| 783/783 [01:22<00:00,  9.46it/s]


Validation Performance
mAP (IoU=0.50:0.95): 0.8642
mAP@50: 0.9688
mAP@75: 0.9645
모든 학습이 완료됨
